# Token Embedding Dynamics

How token representations evolve through the model: identity evolution,
embedding similarity, mixing rate, distances, and prediction trajectory.

## Why This Matters

Token embedding dynamics tracks how individual token representations evolve through the network. Understanding token-level dynamics reveals how context is integrated, when predictions form, and how different positions interact to produce the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_embedding_dynamics import (
    token_identity_evolution, embedding_residual_similarity,
    token_mixing_rate, token_representation_distance,
    token_prediction_trajectory,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Token Identity Evolution

Does the model retain the input token's identity through layers?

In [ ]:
result = token_identity_evolution(model, tokens, position=-1)
print(f"Input token: {result['input_token']}\n")
for p in result['per_layer']:
    retain = '✓' if p['retains_identity'] else '✗'
    print(f"  Layer {p['layer']}: input_prob={p['input_token_prob']:.4f}, "
          f"rank={p['input_token_rank']}, top={p['top_prediction']} [{retain}]")

## Embedding-Residual Similarity

How similar is each position's residual to its original embedding?

In [ ]:
result = embedding_residual_similarity(model, tokens)
for p in result['per_position']:
    sims = ' → '.join(f"{l['cosine_to_embed']:+.3f}" for l in p['per_layer'])
    print(f"  Pos {p['position']} (tok {p['token']:2d}): {sims} → final={p['final_similarity']:+.4f}")

## Token Mixing Rate

How quickly do representations diverge from their initial embeddings?

In [ ]:
result = token_mixing_rate(model, tokens)
for p in result['per_layer']:
    bar = '█' * int(max(0, p['mean_embed_similarity']) * 20)
    print(f"  Layer {p['layer']}: mean_sim={p['mean_embed_similarity']:+.4f}, "
          f"range=[{p['min_similarity']:+.3f}, {p['max_similarity']:+.3f}] {bar}")

## Representation Distance

Pairwise distances between token representations at each layer.

In [ ]:
for layer in range(4):
    result = token_representation_distance(model, tokens, layer=layer)
    print(f"Layer {layer}: mean_sim={result['mean_similarity']:.4f}")
    for p in sorted(result['pairs'], key=lambda x: x['cosine'], reverse=True)[:3]:
        print(f"  pos{p['pos_a']}↔pos{p['pos_b']}: cos={p['cosine']:+.4f}")
    print()

## Prediction Trajectory

How does the top prediction change layer by layer?

In [ ]:
result = token_prediction_trajectory(model, tokens, position=-1)
print(f"Prediction changes: {result['n_changes']}\n")
for p in result['per_layer']:
    changed = ' [CHANGED]' if p['top_changed'] else ''
    top3 = ', '.join(f"tok{t['token']}({t['prob']:.3f})" for t in p['top_predictions'][:3])
    print(f"  Layer {p['layer']}: {top3}{changed}")